In [ ]:
# !pip install ninja --break-system-packages

In [ ]:
# !pip install --upgrade transformers --break-system-packages

In [ ]:
# !pip install git+https://github.com/intel/auto-round.git --break-system-packages

In [ ]:
# !pip install git+https://github.com/sustcsonglin/flash-linear-attention.git --no-build-isolation --break-system-packages

In [ ]:
# !pip install compressed-tensors --break-system-packages

In [1]:
import os

import torch
from auto_round import AutoRound
from huggingface_hub import HfApi, create_repo, get_token, notebook_login
from transformers import AutoModelForImageTextToText, AutoProcessor


In [2]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [3]:
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")


PyTorch Version: 2.14.0+cu126
CUDA Available: True
CUDA Version: 12.6
GPU Name: NVIDIA H200
VRAM: 139.8 GB


In [4]:
notebook_login()

In [5]:
MODEL_ID = "Qwen/Qwen3.5-9B"
HF_USER = "Vishva007"
OUTPUT_BASE_DIR = "./AutoRound"
LOCAL_PATH = "./local_model"

In [6]:
!hf download $MODEL_ID --local-dir $LOCAL_PATH

Reconstructing (incomplete total...): |           |  0.00B /  0.00B            

Fetching 16 files: 100%|██████████████████████| 16/16 [00:00<00:00, 427.31it/s]
Download complete: :                                       |  0.00B            
Reconstruction complete: |                        |  0.00B /  0.00B            ✓ Downloaded
  path: /workspace/local_model
Download complete: :                                       |  0.00B            
Reconstruction complete: |                        |  0.00B /  0.00B            


In [7]:
import os

from safetensors import safe_open

for file in os.listdir(LOCAL_PATH):
    if file.endswith(".safetensors"):
        path = os.path.join(LOCAL_PATH, file)
        print(f"\nChecking {file}")

        with safe_open(path, framework="pt") as f:
            keys = list(f.keys())

            mtp_keys = [k for k in keys if "mtp" in k.lower()]
            for k in mtp_keys:
                print(k)


Checking model.safetensors-00002-of-00004.safetensors
mtp.layers.0.mlp.down_proj.weight
mtp.layers.0.mlp.gate_proj.weight
mtp.layers.0.mlp.up_proj.weight

Checking model.safetensors-00003-of-00004.safetensors
mtp.fc.weight
mtp.layers.0.self_attn.q_proj.weight

Checking model.safetensors-00001-of-00004.safetensors

Checking model.safetensors-00004-of-00004.safetensors
mtp.layers.0.input_layernorm.weight
mtp.layers.0.post_attention_layernorm.weight
mtp.layers.0.self_attn.k_norm.weight
mtp.layers.0.self_attn.k_proj.weight
mtp.layers.0.self_attn.o_proj.weight
mtp.layers.0.self_attn.q_norm.weight
mtp.layers.0.self_attn.v_proj.weight
mtp.norm.weight
mtp.pre_fc_norm_embedding.weight
mtp.pre_fc_norm_hidden.weight


In [8]:
model = AutoModelForImageTextToText.from_pretrained(
    LOCAL_PATH, 
    dtype=torch.bfloat16, 
    device_map="auto"
)
processor = AutoProcessor.from_pretrained(LOCAL_PATH)

tokenizer = processor.tokenizer


Loading weights:   0%|          | 0/760 [00:00<?, ?it/s]

In [9]:
model

Qwen3_5ForConditionalGeneration(
  (model): Qwen3_5Model(
    (visual): Qwen3_5VisionModel(
      (patch_embed): Qwen3_5VisionPatchEmbed(
        (proj): Conv3d(3, 1152, kernel_size=(2, 16, 16), stride=(2, 16, 16))
      )
      (pos_embed): Embedding(2304, 1152)
      (rotary_pos_emb): Qwen3_5VisionRotaryEmbedding()
      (blocks): ModuleList(
        (0-26): 27 x Qwen3_5VisionBlock(
          (norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True, bias=True)
          (norm2): LayerNorm((1152,), eps=1e-06, elementwise_affine=True, bias=True)
          (attn): Qwen3_5VisionAttention(
            (qkv): Linear(in_features=1152, out_features=3456, bias=True)
            (proj): Linear(in_features=1152, out_features=1152, bias=True)
          )
          (mlp): Qwen3_5VisionMLP(
            (linear_fc1): Linear(in_features=1152, out_features=4304, bias=True)
            (linear_fc2): Linear(in_features=4304, out_features=1152, bias=True)
            (act_fn): GELUTanh()
         

In [16]:
def push_to_hub(local_dir, repo_name, token):
    """Delete repo and create fresh one."""
    full_repo_id = f"{HF_USER}/{repo_name}"
    
    try:
        api = HfApi()
        
        # Delete repo (WARNING: irreversible!)
        print(f"[Hub] Deleting existing repo...")
        api.delete_repo(repo_id=full_repo_id, repo_type="model", token=token)
        
        # Create fresh repo
        print(f"[Hub] Creating new repo...")
        create_repo(full_repo_id, repo_type="model", private=False, token=token)
        
        # Upload everything
        print(f"[Hub] Uploading...")
        api.upload_folder(
            folder_path=local_dir,
            repo_id=full_repo_id,
            repo_type="model",
            token=token
        )
        print(f"[Hub] ✅ Successfully uploaded: https://huggingface.co/{full_repo_id}")
        
    except Exception as e:
        print(f"[Hub] ❌ Error: {e}")

In [ ]:
TUNING_CONFIG = {
    "group_size": 32,
    "sym": True,
    "iters": 1200,  # High accuracy (Production grade)
    "nsamples": 512,  # More calibration data
    "batch_size": 4,
    "seqlen": 4096,
    "low_gpu_mem_usage": False,  # Keep on GPU for speed
    "enable_torch_compile": True,  # JIT acceleration
    "quant_nontext_module": False,  # Keep Vision Tower in FP16 (Crucial for VLM accuracy)
    "layer_config": {
        "mtp": {"data_type": "bfloat16"},
        "mtp.fc": {"data_type": "bfloat16"}
    }
}

In [12]:
ar = AutoRound(
    model=model,
    tokenizer=tokenizer,
    processor=processor,
    scheme="W4A16",
    **TUNING_CONFIG,
)

2026-09-20 03:00:26 WARNING autoround.py L489: Passing 'group_size' directly to AutoRound is supported, but the recommended usage is 'alg_configs=SignRoundConfig(...)'.
2026-09-20 03:00:26 WARNING autoround.py L489: Passing 'sym' directly to AutoRound is supported, but the recommended usage is 'alg_configs=SignRoundConfig(...)'.
2026-09-20 03:00:26 WARNING autoround.py L489: Passing 'iters' directly to AutoRound is supported, but the recommended usage is 'alg_configs=SignRoundConfig(...)'.


In [13]:
# SINGLE CALL to save all 3 formats to the same output directory
# The files will exist side-by-side or merged in this folder.
ar.quantize_and_save(
    OUTPUT_BASE_DIR, format="auto_round,auto_gptq,llm_compressor", inplace=True
)

2026-09-20 03:00:26 WARNING logging.py L340: some layers are skipped quantization (shape not divisible by 32): model.visual.blocks.[0-26].mlp.linear_fc1, model.visual.blocks.[0-26].mlp.linear_fc2
2026-09-20 03:00:27 INFO base.py L1366: `torch.compile` is enabled
2026-09-20 03:00:27 INFO orchestrator.py L594: start to cache block inputs
2026-09-20 03:00:27 INFO mllm.py L86: Using MLLM template: qwen3_5
2026-09-20 03:00:27 INFO mllm.py L125: Multimodal model with non-MLLM calibration dataset 'NeelNanda/pile-10k' and quant_nontext_module=False: using the standard text dataloader (vision/audio towers are not being quantized, so text-only calibration through the full-model forward is sufficient).
2026-09-20 03:00:27 INFO calib_dataset.py L1213: Preprocessing calibration dataset in a subprocess to avoid memory leaks...
2026-09-20 03:00:51 INFO device.py L1560: 'peak_ram': 35.42GB, 'peak_vram': 17.69GB
2026-09-20 03:00:51 INFO orchestrator.py L626: caching done
Quantizing model.language_model

Writing model shards:   0%|          | 0/2 [00:00<?, ?it/s]

2026-09-20 03:35:54 INFO missing_tensors.py L125: Restored 48 tensor(s) from FP16/BF16 to their original FP32 values: model.language_model.layers.[0-2,4-6,8-10,12-14,16-18,20-22,24-26,28-30].linear_attn, model.language_model.layers.[0-2,4-6,8-10,12-14,16-18,20-22,24-26,28-30].linear_attn.norm.
2026-09-20 03:35:54 INFO missing_tensors.py L306: Found 15 tensor(s) in the source checkpoint that are absent from the saved output (e.g., MTP parameters): mtp.fc, mtp.layers.0.input_layernorm, mtp.layers.0.mlp.down_proj, mtp.layers.0.mlp.gate_proj, mtp.layers.0.mlp.up_proj, mtp.layers.0.post_attention_layernorm, mtp.layers.0.self_attn.k_norm, mtp.layers.0.self_attn.k_proj, mtp.layers.0.self_attn.o_proj, mtp.layers.0.self_attn.q_norm, mtp.layers.0.self_attn.q_proj, mtp.layers.0.self_attn.v_proj, mtp.norm, mtp.pre_fc_norm_embedding, mtp.pre_fc_norm_hidden. Copying them now...

Loading missing tensors: 100%|██████████| 3/3 [00:00<00:00,  3.86shard/s]
2026-09-20 03:35:55 INFO missing_tensors.py L657

Writing model shards:   0%|          | 0/2 [00:00<?, ?it/s]

2026-09-20 03:38:47 INFO missing_tensors.py L125: Restored 48 tensor(s) from FP16/BF16 to their original FP32 values: model.language_model.layers.[0-2,4-6,8-10,12-14,16-18,20-22,24-26,28-30].linear_attn, model.language_model.layers.[0-2,4-6,8-10,12-14,16-18,20-22,24-26,28-30].linear_attn.norm.
2026-09-20 03:38:47 INFO missing_tensors.py L306: Found 15 tensor(s) in the source checkpoint that are absent from the saved output (e.g., MTP parameters): mtp.fc, mtp.layers.0.input_layernorm, mtp.layers.0.mlp.down_proj, mtp.layers.0.mlp.gate_proj, mtp.layers.0.mlp.up_proj, mtp.layers.0.post_attention_layernorm, mtp.layers.0.self_attn.k_norm, mtp.layers.0.self_attn.k_proj, mtp.layers.0.self_attn.o_proj, mtp.layers.0.self_attn.q_norm, mtp.layers.0.self_attn.q_proj, mtp.layers.0.self_attn.v_proj, mtp.norm, mtp.pre_fc_norm_embedding, mtp.pre_fc_norm_hidden. Copying them now...

Loading missing tensors: 100%|██████████| 3/3 [00:00<00:00, 147.35shard/s]
2026-09-20 03:38:48 INFO missing_tensors.py L44

Writing model shards:   0%|          | 0/2 [00:00<?, ?it/s]

2026-09-20 03:44:50 INFO missing_tensors.py L125: Restored 48 tensor(s) from FP16/BF16 to their original FP32 values: model.language_model.layers.[0-2,4-6,8-10,12-14,16-18,20-22,24-26,28-30].linear_attn, model.language_model.layers.[0-2,4-6,8-10,12-14,16-18,20-22,24-26,28-30].linear_attn.norm.
2026-09-20 03:44:50 INFO missing_tensors.py L306: Found 15 tensor(s) in the source checkpoint that are absent from the saved output (e.g., MTP parameters): mtp.fc, mtp.layers.0.input_layernorm, mtp.layers.0.mlp.down_proj, mtp.layers.0.mlp.gate_proj, mtp.layers.0.mlp.up_proj, mtp.layers.0.post_attention_layernorm, mtp.layers.0.self_attn.k_norm, mtp.layers.0.self_attn.k_proj, mtp.layers.0.self_attn.o_proj, mtp.layers.0.self_attn.q_norm, mtp.layers.0.self_attn.q_proj, mtp.layers.0.self_attn.v_proj, mtp.norm, mtp.pre_fc_norm_embedding, mtp.pre_fc_norm_hidden. Copying them now...

Loading missing tensors: 100%|██████████| 3/3 [00:00<00:00, 146.39shard/s]
2026-09-20 03:44:51 INFO missing_tensors.py L44

(Qwen3_5ForConditionalGeneration(
   (model): Qwen3_5Model(
     (visual): Qwen3_5VisionModel(
       (patch_embed): Qwen3_5VisionPatchEmbed(
         (proj): Conv3d(3, 1152, kernel_size=(2, 16, 16), stride=(2, 16, 16))
       )
       (pos_embed): Embedding(2304, 1152)
       (rotary_pos_emb): Qwen3_5VisionRotaryEmbedding()
       (blocks): ModuleList(
         (0-26): 27 x Qwen3_5VisionBlock(
           (norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True, bias=True)
           (norm2): LayerNorm((1152,), eps=1e-06, elementwise_affine=True, bias=True)
           (attn): Qwen3_5VisionAttention(
             (qkv): Linear(in_features=1152, out_features=3456, bias=True)
             (proj): Linear(in_features=1152, out_features=1152, bias=True)
           )
           (mlp): Qwen3_5VisionMLP(
             (linear_fc1): Linear(in_features=1152, out_features=4304, bias=True)
             (linear_fc2): Linear(in_features=4304, out_features=1152, bias=True)
             (act_fn): 

In [14]:
base_name = MODEL_ID.split("/")[-1]
hf_token = get_token()

In [17]:
if hf_token:
    push_to_hub(
        os.path.join(OUTPUT_BASE_DIR, "local_model-w4g32/auto-round-auto-gptq"), 
        f"{base_name}-W4A16-AutoRound", 
        hf_token)
    push_to_hub(
        os.path.join(OUTPUT_BASE_DIR, "local_model-w4g32/auto-gptq"), 
        f"{base_name}-W4A16-AutoRound-GPTQ",
        hf_token
    )
    push_to_hub(
            os.path.join(OUTPUT_BASE_DIR, "local_model-w4g32/llm-compressor-wint-a16"), 
            f"{base_name}-W4A16-AutoRound-LLM-Compressor", 
            hf_token)
else:
    print("No Hugging Face token found. Skipping upload to hub.")

[Hub] Deleting existing repo...


[Hub] Creating new repo...
[Hub] Uploading...
[Hub] ✅ Successfully uploaded: https://huggingface.co/Vishva007/Qwen3.5-9B-W4A16-AutoRound
[Hub] Deleting existing repo...
[Hub] Creating new repo...
[Hub] Uploading...
[Hub] ✅ Successfully uploaded: https://huggingface.co/Vishva007/Qwen3.5-9B-W4A16-AutoRound-GPTQ
[Hub] Deleting existing repo...
[Hub] Creating new repo...
[Hub] Uploading...
[Hub] ✅ Successfully uploaded: https://huggingface.co/Vishva007/Qwen3.5-9B-W4A16-AutoRound-LLM-Compressor
